In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [43]:
cfa = pd.read_csv(r"Customer Flight Activity (1).csv")
clh = pd.read_csv(r"C:\Users\Dev\OneDrive\Desktop\IIT Guwahati\Customer Loyalty History (1).csv")
cal = pd.read_csv(r"C:\Users\Dev\OneDrive\Desktop\IIT Guwahati\Calendar (1).csv")   

cfa = cfa.drop_duplicates()
clh = clh.drop_duplicates()
cal = cal.drop_duplicates()


print(cfa.shape)
print(clh.shape)

(391014, 8)
(16737, 16)


In [59]:
clh['Salary'][(clh['Salary'] > 0 ) & (clh['Education'] == 'Bachelor' )].describe()



count     10456.000000
mean      72645.926262
std       16590.634159
min       15609.000000
25%       58715.500000
50%       72026.000000
75%       85848.000000
max      105563.000000
Name: Salary, dtype: float64

In [ ]:
# Median Imputation to all the negative salary
clh.loc[:,'Salary'][(clh['Salary'] <0 ) ] = 72026

C:\Users\Dev\AppData\Local\Temp\ipykernel_26588\1106392178.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  clh.loc[:,'Salary'][(clh['Salary'] <0 ) ] = 72026
C:\Users\Dev\AppData\Local\Temp\ipykernel_26588\1106392178.py:1: SettingWithCopy

In [62]:
clh.loc[:,['Education','Salary']][(clh['Salary']  < 0) ] = (-1)*clh.loc[:,['Education','Salary']][(clh['Salary']  < 0) ]

In [63]:
clh[(clh['Education'] == "College")]

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,2018.0,1.0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,NaN,NaN
6,927943,Canada,Ontario,Toronto,P5S 6R4,Female,College,NaN,Single,Star,3857.95,Standard,2014,6,NaN,NaN
13,988178,Canada,Quebec,Montreal,H4G 3T4,Male,College,NaN,Single,Star,3871.07,Standard,2013,10,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16721,632951,Canada,Alberta,Edmonton,T9G 1W3,Female,College,NaN,Married,Star,44771.30,Standard,2018,7,NaN,NaN
16727,546773,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Married,Star,52811.49,Standard,2015,9,NaN,NaN
16731,900501,Canada,Ontario,Sudbury,M5V 1G5,Male,College,NaN,Single,Star,61134.68,Standard,2012,9,NaN,NaN
16732,823768,Canada,British Columbia,Vancouver,V6E 3Z3,Female,College,NaN,Married,Star,61850.19,Standard,2012,12,NaN,NaN


### Step 2 — Salary Null Handling

**Issue:** NaN salary values present for College students.  
**Decision:** Filled with 0 for College students.  
**Reason:** College students have no income — 0 is factually 
correct, not an imputation. Confirmed via Education column.

In [64]:
# Fill NaN salary with 0 for College students
clh.loc[clh['Education'] == 'College', 'Salary'] = clh.loc[
    clh['Education'] == 'College', 'Salary'].fillna(0)

# Verify
print("Null salaries remaining:", clh['Salary'].isna().sum())
print("College students with 0 salary:", 
      ((clh['Education'] == 'College') & (clh['Salary'] == 0)).sum())

Null salaries remaining: 0
College students with 0 salary: 4238


In [65]:
print(clh.info())
clh.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16737 entries, 0 to 16736
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Loyalty Number      16737 non-null  int64  
 1   Country             16737 non-null  object 
 2   Province            16737 non-null  object 
 3   City                16737 non-null  object 
 4   Postal Code         16737 non-null  object 
 5   Gender              16737 non-null  object 
 6   Education           16737 non-null  object 
 7   Salary              16737 non-null  float64
 8   Marital Status      16737 non-null  object 
 9   Loyalty Card        16737 non-null  object 
 10  CLV                 16737 non-null  float64
 11  Enrollment Type     16737 non-null  object 
 12  Enrollment Year     16737 non-null  int64  
 13  Enrollment Month    16737 non-null  int64  
 14  Cancellation Year   2067 non-null   float64
 15  Cancellation Month  2067 non-null   float64
dtypes: f

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,0.0,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,0.0,Single,Star,3839.75,Standard,2014,7,2018.0,1.0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,0.0,Single,Star,3839.75,Standard,2013,2,NaN,NaN
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,NaN,NaN


### Step 3 — Build Master Dataset

Merged Customer Flight Activity and Customer Loyalty History 
on Loyalty Number using inner join.

- Inner join used — only keep customers present in both files
- Result: one row per customer per month
- 392,936 rows × 25 columns

In [66]:
# Merge flight activity + loyalty history on Loyalty Number
master = cfa.merge(clh, on='Loyalty Number', how='inner')

# Verify
print(f"Master shape: {master.shape}")
print(f"Unique members: {master['Loyalty Number'].nunique():,}")

master.sample(5)

Master shape: (391014, 23)
Unique members: 16,737


,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,Country,Province,...,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
232673,925609,2017,6,0,0,0.0,0,0,Canada,Quebec,...,College,0.0,Divorced,Star,2659.71,Standard,2016,7,NaN,NaN
149902,141844,2017,12,1,1428,1428.0,0,0,Canada,British Columbia,...,Bachelor,55552.0,Married,Nova,8109.36,Standard,2012,10,NaN,NaN
86305,200863,2018,11,3,2121,2121.0,0,0,Canada,Quebec,...,Bachelor,96102.0,Divorced,Star,8432.52,Standard,2016,7,NaN,NaN
188245,227293,2017,2,0,0,0.0,0,0,Canada,British Columbia,...,Bachelor,74727.0,Married,Aurora,10899.30,Standard,2013,1,NaN,NaN
301348,798351,2018,2,0,0,0.0,0,0,Canada,British Columbia,...,College,0.0,Single,Star,2144.92,Standard,2012,5,NaN,NaN


In [67]:
master[master["Loyalty Number"] == 100590]

,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,Country,Province,...,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,100590,2018,6,12,15276,22914.0,0,0,Canada,British Columbia,...,Bachelor,69841.0,Married,Star,34090.04,2018 Promotion,2018,3,NaN,NaN
1,100590,2018,7,12,9168,13752.0,0,0,Canada,British Columbia,...,Bachelor,69841.0,Married,Star,34090.04,2018 Promotion,2018,3,NaN,NaN
2,100590,2018,5,4,6504,9756.0,0,0,Canada,British Columbia,...,Bachelor,69841.0,Married,Star,34090.04,2018 Promotion,2018,3,NaN,NaN
3,100590,2018,10,0,0,0.0,512,92,Canada,British Columbia,...,Bachelor,69841.0,Married,Star,34090.04,2018 Promotion,2018,3,NaN,NaN
4,100590,2018,2,0,0,0.0,0,0,Canada,British Columbia,...,Bachelor,69841.0,Married,Star,34090.04,2018 Promotion,2018,3,NaN,NaN
5,100590,2018,4,0,0,0.0,0,0,Canada,British Columbia,...,Bachelor,69841.0,Married,Star,34090.04,2018 Promotion,2018,3,NaN,NaN
6,100590,2018,3,0,0,0.0,0,0,Canada,British Columbia,...,Bachelor,69841.0,Married,Star,34090.04,2018 Promotion,2018,3,NaN,NaN
7,100590,2018,8,0,0,0.0,0,0,Canada,British Columbia,...,Bachelor,69841.0,Married,Star,34090.04,2018 Promotion,2018,3,NaN,NaN
8,100590,2018,9,0,0,0.0,0,0,Canada,British Columbia,...,Bachelor,69841.0,Married,Star,34090.04,2018 Promotion,2018,3,NaN,NaN
9,100590,2018,11,0,0,0.0,0,0,Canada,British Columbia,...,Bachelor,69841.0,Married,Star,34090.04,2018 Promotion,2018,3,NaN,NaN


# Sort the merged data by Loyalty Number , Year and Month and then reset index

In [68]:
master = master.sort_values(ascending=True , by= ['Loyalty Number','Year','Month']).reset_index(drop = True)
master.head(25)

,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,Country,Province,...,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,100018,2017,1,1,601,601.0,0,0,Canada,Alberta,...,Bachelor,92552.0,Married,Aurora,7919.20,Standard,2016,8,NaN,NaN
1,100018,2017,2,0,0,0.0,0,0,Canada,Alberta,...,Bachelor,92552.0,Married,Aurora,7919.20,Standard,2016,8,NaN,NaN
2,100018,2017,3,4,9648,9648.0,438,79,Canada,Alberta,...,Bachelor,92552.0,Married,Aurora,7919.20,Standard,2016,8,NaN,NaN
3,100018,2017,4,1,1654,1654.0,0,0,Canada,Alberta,...,Bachelor,92552.0,Married,Aurora,7919.20,Standard,2016,8,NaN,NaN
4,100018,2017,5,0,0,0.0,0,0,Canada,Alberta,...,Bachelor,92552.0,Married,Aurora,7919.20,Standard,2016,8,NaN,NaN
5,100018,2017,6,1,2489,2489.0,0,0,Canada,Alberta,...,Bachelor,92552.0,Married,Aurora,7919.20,Standard,2016,8,NaN,NaN
6,100018,2017,7,3,3687,3687.0,0,0,Canada,Alberta,...,Bachelor,92552.0,Married,Aurora,7919.20,Standard,2016,8,NaN,NaN
7,100018,2017,8,3,7116,7116.0,690,124,Canada,Alberta,...,Bachelor,92552.0,Married,Aurora,7919.20,Standard,2016,8,NaN,NaN
8,100018,2017,9,3,5751,5751.0,0,0,Canada,Alberta,...,Bachelor,92552.0,Married,Aurora,7919.20,Standard,2016,8,NaN,NaN
9,100018,2017,10,2,3728,3728.0,0,0,Canada,Alberta,...,Bachelor,92552.0,Married,Aurora,7919.20,Standard,2016,8,NaN,NaN


In [69]:
# Verify
print(f"Master shape: {master.shape}")
print(f"Unique members: {master['Loyalty Number'].nunique():,}")

Master shape: (391014, 23)
Unique members: 16,737


In [70]:
master[master['Loyalty Number'] == 201574]

,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,Country,Province,...,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
43476,201574,2017,1,0,0,0.0,0,0,Canada,Ontario,...,Bachelor,51375.0,Married,Star,3893.31,Standard,2015,4,2015.0,12.0
43477,201574,2017,2,0,0,0.0,0,0,Canada,Ontario,...,Bachelor,51375.0,Married,Star,3893.31,Standard,2015,4,2015.0,12.0
43478,201574,2017,3,0,0,0.0,0,0,Canada,Ontario,...,Bachelor,51375.0,Married,Star,3893.31,Standard,2015,4,2015.0,12.0
43479,201574,2017,4,0,0,0.0,0,0,Canada,Ontario,...,Bachelor,51375.0,Married,Star,3893.31,Standard,2015,4,2015.0,12.0
43480,201574,2017,5,0,0,0.0,0,0,Canada,Ontario,...,Bachelor,51375.0,Married,Star,3893.31,Standard,2015,4,2015.0,12.0
43481,201574,2017,6,0,0,0.0,0,0,Canada,Ontario,...,Bachelor,51375.0,Married,Star,3893.31,Standard,2015,4,2015.0,12.0
43482,201574,2017,7,0,0,0.0,0,0,Canada,Ontario,...,Bachelor,51375.0,Married,Star,3893.31,Standard,2015,4,2015.0,12.0
43483,201574,2017,8,0,0,0.0,0,0,Canada,Ontario,...,Bachelor,51375.0,Married,Star,3893.31,Standard,2015,4,2015.0,12.0
43484,201574,2017,9,0,0,0.0,0,0,Canada,Ontario,...,Bachelor,51375.0,Married,Star,3893.31,Standard,2015,4,2015.0,12.0
43485,201574,2017,10,0,0,0.0,0,0,Canada,Ontario,...,Bachelor,51375.0,Married,Star,3893.31,Standard,2015,4,2015.0,12.0


### Step 4 — Derive Quarter and Season

Quarter and Season derived directly from Month column.
Canadian seasonal mapping used:
- Winter : Dec, Jan, Feb
- Spring : Mar, Apr, May  
- Summer : Jun, Jul, Aug
- Fall   : Sep, Oct, Nov

Calendar file not used — same information derived 
directly from Month in two lines.

In [72]:
# Derive Quarter
master['Quarter'] = master['Month'].apply(lambda m: f'Q{(m-1) // 3 + 1}')

# Derive Season — Canadian context
season_map = {
    1: 'Winter', 2: 'Winter',  3: 'Spring',
    4: 'Spring', 5: 'Spring',  6: 'Summer',
    7: 'Summer', 8: 'Summer',  9: 'Fall',
    10: 'Fall',  11: 'Fall',   12: 'Winter'
}
master['Season'] = master['Month'].map(season_map)

# Verify
print(master[['Month', 'Quarter', 'Season']].drop_duplicates().sort_values('Month'))

    Month Quarter  Season
0       1      Q1  Winter
1       2      Q1  Winter
2       3      Q1  Spring
3       4      Q2  Spring
4       5      Q2  Spring
5       6      Q2  Summer
6       7      Q3  Summer
7       8      Q3  Summer
8       9      Q3    Fall
9      10      Q4    Fall
10     11      Q4    Fall
11     12      Q4  Winter


### Step 5 — Save Master Dataset

Cleaned merged dataset saved as master.csv.
- 392,936 rows × 25 columns
- One row per customer per month
- All cleaning fixes applied
- Quarter and Season columns added
- Ready for feature engineering

In [73]:
# Save master
master.to_csv('master.csv', index=False)

# Verify
print(f"Master saved successfully")
print(f"Shape: {master.shape}")
print(f"Columns: {list(master.columns)}")

Master saved successfully
Shape: (391014, 25)
Columns: ['Loyalty Number', 'Year', 'Month', 'Total Flights', 'Distance', 'Points Accumulated', 'Points Redeemed', 'Dollar Cost Points Redeemed', 'Country', 'Province', 'City', 'Postal Code', 'Gender', 'Education', 'Salary', 'Marital Status', 'Loyalty Card', 'CLV', 'Enrollment Type', 'Enrollment Year', 'Enrollment Month', 'Cancellation Year', 'Cancellation Month', 'Quarter', 'Season']


In [74]:
# Add Quarter and Season
master['Quarter'] = master['Month'].apply(lambda m: f'Q{(m-1) // 3 + 1}')

season_map = {
    1: 'Winter', 2: 'Winter',  3: 'Spring',
    4: 'Spring', 5: 'Spring',  6: 'Summer',
    7: 'Summer', 8: 'Summer',  9: 'Fall',
    10: 'Fall',  11: 'Fall',   12: 'Winter'
}
master['Season'] = master['Month'].map(season_map)

# Verify
print(master.shape)
print(list(master.columns))

# Save
master.to_csv('master.csv', index=False)
print("Master saved")

(391014, 25)
['Loyalty Number', 'Year', 'Month', 'Total Flights', 'Distance', 'Points Accumulated', 'Points Redeemed', 'Dollar Cost Points Redeemed', 'Country', 'Province', 'City', 'Postal Code', 'Gender', 'Education', 'Salary', 'Marital Status', 'Loyalty Card', 'CLV', 'Enrollment Type', 'Enrollment Year', 'Enrollment Month', 'Cancellation Year', 'Cancellation Month', 'Quarter', 'Season']
Master saved


### Step 6 — Formal Churn Label

**Definition 1 — Formal Churn:**
A customer is formally churned if Cancellation Year 
is recorded in the loyalty history.

formally_cancelled = 1 → customer cancelled
formally_cancelled = 0 → never formally cancelled

Churn rate: 12.3% — too narrow to use alone.
87.7% never cancelled but may be behaviorally inactive.

In [75]:
# Build churn label from clh directly
churn_df = clh[['Loyalty Number', 'Cancellation Year']].copy()

# Formal churn
churn_df['formally_cancelled'] = churn_df['Cancellation Year'].notna().astype(int)

# Behavioral churn — zero flights ever
member_flights = cfa.groupby('Loyalty Number')['Total Flights'].sum()
churn_df['zero_flights_ever'] = churn_df['Loyalty Number'].map(
    member_flights).fillna(0) == 0
churn_df['zero_flights_ever'] = churn_df['zero_flights_ever'].astype(int)

# Final combined label
churn_df['churned'] = ((churn_df['formally_cancelled'] == 1) |
                       (churn_df['zero_flights_ever'] == 1)).astype(int)

# Keep only what's needed
churn_df = churn_df[['Loyalty Number', 'churned']]

# Verify
print(churn_df.shape)
print(churn_df['churned'].value_counts())
print(f"Churn rate: {churn_df['churned'].mean()*100:.1f}%")

(16737, 2)
churned
0    14051
1     2686
Name: count, dtype: int64
Churn rate: 16.0%


In [76]:
# Membership tenure
clh['Tenure_Months'] = (
    (2018 - clh['Enrollment Year']) * 12 +
    (12 - clh['Enrollment Month'])
)

# Select demographic features
demographic = clh[[
    'Loyalty Number',
    'Gender',
    'Education',
    'Salary',
    'Marital Status',
    'Loyalty Card',
    'CLV',
    'Enrollment Type',
    'Province',
    'Tenure_Months'
]].copy()

# Verify
print(demographic.shape)
print(demographic.isnull().sum())

(16737, 10)
Loyalty Number     0
Gender             0
Education          0
Salary             0
Marital Status     0
Loyalty Card       0
CLV                0
Enrollment Type    0
Province           0
Tenure_Months      0
dtype: int64


In [77]:
# Flight volume features
flight_features = cfa.groupby('Loyalty Number').agg(
    Total_Flights        = ('Total Flights', 'sum'),
    Avg_Flights_Month    = ('Total Flights', 'mean'),
    Max_Flights_Month    = ('Total Flights', 'max'),
    Total_Distance       = ('Distance', 'sum'),
    Avg_Distance_Month   = ('Distance', 'mean')
).reset_index()

# Points features
points_features = cfa.groupby('Loyalty Number').agg(
    Total_Points_Acc  = ('Points Accumulated', 'sum'),
    Total_Points_Red  = ('Points Redeemed', 'sum'),
    Total_Dollar_Red  = ('Dollar Cost Points Redeemed', 'sum')
).reset_index()

# Redemption rate
points_features['Redemption_Rate'] = (
    points_features['Total_Points_Red'] /
    points_features['Total_Points_Acc'].replace(0, 1)
)

# Verify
print(flight_features.shape)
print(points_features.shape)

(16737, 6)
(16737, 5)


In [78]:
# Sequential period index — 2017 Jan=1 to 2018 Dec=24
cfa['Period'] = (cfa['Year'] - 2017) * 12 + cfa['Month']

# Last active period per customer
last_active = cfa[cfa['Total Flights'] > 0].groupby(
    'Loyalty Number')['Period'].max().reset_index()
last_active.columns = ['Loyalty Number', 'Last_Active_Period']

# Months since last flight
max_period = cfa['Period'].max()
last_active['Months_Since_Last_Flight'] = max_period - last_active['Last_Active_Period']

# Members with zero flights get maximum recency
all_members = pd.DataFrame({'Loyalty Number': cfa['Loyalty Number'].unique()})
recency = all_members.merge(last_active, on='Loyalty Number', how='left')
recency['Months_Since_Last_Flight'] = recency['Months_Since_Last_Flight'].fillna(max_period)
recency = recency[['Loyalty Number', 'Months_Since_Last_Flight']]

# Zero activity months
zero_months = cfa[cfa['Total Flights'] == 0].groupby(
    'Loyalty Number')['Month'].count().reset_index()
zero_months.columns = ['Loyalty Number', 'Zero_Activity_Months']

# Verify
print(f"Max period: {max_period}")
print(recency.shape)
print(recency['Months_Since_Last_Flight'].describe())

Max period: 24
(16737, 2)
count    16737.000000
mean         3.561929
std          7.375618
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max         24.000000
Name: Months_Since_Last_Flight, dtype: float64


In [79]:
# Flights per season per customer
season_flights = master.groupby(
    ['Loyalty Number', 'Season'])['Total Flights'].sum().unstack(fill_value=0)

# Rename columns
season_flights.columns = [f'Flights_{s}' for s in season_flights.columns]
season_flights = season_flights.reset_index()

# Verify
print(season_flights.shape)
print(season_flights.head(3))

(16737, 5)
   Loyalty Number  Flights_Fall  Flights_Spring  Flights_Summer  \
0          100018            18               8              10   
1          100102            16              13              10   
2          100140            13              13              11   

   Flights_Winter  
0              10  
1              12  
2              10  


In [94]:
# Start with demographic features
feature_matrix = demographic.copy()

# Add churn label
feature_matrix = feature_matrix.merge(churn_df, on='Loyalty Number', how='left')

# Add flight features
feature_matrix = feature_matrix.merge(flight_features, on='Loyalty Number', how='left')

# Add points features
feature_matrix = feature_matrix.merge(points_features, on='Loyalty Number', how='left')

# Add recency features
feature_matrix = feature_matrix.merge(recency, on='Loyalty Number', how='left')

# Add zero activity months
feature_matrix = feature_matrix.merge(zero_months, on='Loyalty Number', how='left')

# Add seasonal features
feature_matrix = feature_matrix.merge(season_flights, on='Loyalty Number', how='left')

# Fill remaining nulls with 0
feature_matrix = feature_matrix.fillna(0)

# Verify
print(f"Shape: {feature_matrix.shape}")
print(f"Nulls: {feature_matrix.isnull().sum().sum()}")
print(f"Columns: {list(feature_matrix.columns)}")

Shape: (16737, 26)
Nulls: 0
Columns: ['Loyalty Number', 'Gender', 'Education', 'Salary', 'Marital Status', 'Loyalty Card', 'CLV', 'Enrollment Type', 'Province', 'Tenure_Months', 'churned', 'Total_Flights', 'Avg_Flights_Month', 'Max_Flights_Month', 'Total_Distance', 'Avg_Distance_Month', 'Total_Points_Acc', 'Total_Points_Red', 'Total_Dollar_Red', 'Redemption_Rate', 'Months_Since_Last_Flight', 'Zero_Activity_Months', 'Flights_Fall', 'Flights_Spring', 'Flights_Summer', 'Flights_Winter']


In [99]:
feature_matrix['Region'] = feature_matrix['Province'].map({
                                                            'Ontario'              : 'East',
                                                            'Quebec'               : 'East',
                                                            'New Brunswick'        : 'East',
                                                            'Nova Scotia'          : 'East',
                                                            'Prince Edward Island' : 'East',
                                                            'Newfoundland'         : 'East',
                                                            'Alberta'              : 'West',
                                                            'British Columbia'     : 'West',
                                                            'Manitoba'             : 'West',
                                                            'Saskatchewan'         : 'West',
                                                            'Yukon'                : 'North'
                                                        })

# feature_matrix['Region'] = feature_matrix['Province'].map(region_map)
# print(feature_matrix['Region'].value_counts()))

In [101]:
feature_matrix = feature_matrix.drop('Province',axis=1)

In [104]:
feature_matrix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16737 entries, 0 to 16736
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Loyalty Number            16737 non-null  int64  
 1   Gender                    16737 non-null  object 
 2   Education                 16737 non-null  object 
 3   Salary                    16737 non-null  float64
 4   Marital Status            16737 non-null  object 
 5   Loyalty Card              16737 non-null  object 
 6   CLV                       16737 non-null  float64
 7   Enrollment Type           16737 non-null  object 
 8   Tenure_Months             16737 non-null  int64  
 9   churned                   16737 non-null  int64  
 10  Total_Flights             16737 non-null  int64  
 11  Avg_Flights_Month         16737 non-null  float64
 12  Max_Flights_Month         16737 non-null  int64  
 13  Total_Distance            16737 non-null  int64  
 14  Avg_Di

In [106]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import *

In [ ]:
# label_encoding - loyalty_card,Education 
# one hot - gender, Marital_status,Region , Enrollment Type 

In [109]:
master.Education.unique()
feature_matrix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16737 entries, 0 to 16736
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Loyalty Number            16737 non-null  int64  
 1   Gender                    16737 non-null  object 
 2   Education                 16737 non-null  object 
 3   Salary                    16737 non-null  float64
 4   Marital Status            16737 non-null  object 
 5   Loyalty Card              16737 non-null  object 
 6   CLV                       16737 non-null  float64
 7   Enrollment Type           16737 non-null  object 
 8   Tenure_Months             16737 non-null  int64  
 9   churned                   16737 non-null  int64  
 10  Total_Flights             16737 non-null  int64  
 11  Avg_Flights_Month         16737 non-null  float64
 12  Max_Flights_Month         16737 non-null  int64  
 13  Total_Distance            16737 non-null  int64  
 14  Avg_Di

In [132]:
ct = ColumnTransformer(transformers = [
                                       ('Onehot',OneHotEncoder(drop='first'),[1,4,7,25])])
ct

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Onehot', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature

In [133]:
encoded_ar  =ct.fit_transform(feature_matrix)
enc_f = ct.get_feature_names_out()


In [135]:
df  = feature_matrix
encoded_ar
for i in range(6):
    df[enc_f[i]] = encoded_ar[:,i]


In [ ]:
df = df.drop(['Gender','Marital Status','Enrollment Type','Region'],axis=1)


In [138]:
# label_encoding - loyalty_card,Education 
df['loyalty_card_encoded'] = df['Loyalty Card'].map({'Star':0,
                                                    'Nova':1,
                                                    'Aurora':2})

df['Education_encoded'] = df['Education'].map({'High School or Below':0,
                                                'College':1,
                                                'Bachelor':2,
                                                'Master':3,
                                                'Doctor':4})


In [139]:
df = df.drop(['Loyalty Card','Education'],axis=1)

In [143]:
df= df.drop('Loyalty Number', axis =1 )

In [155]:
# Flight consitstency score
df['Flight_consistency_score'] = df.iloc[:,4]/(df.iloc[:,14]+1)

In [156]:
print(df.head().to_string())

     Salary      CLV  Tenure_Months  churned  Total_Flights  Avg_Flights_Month  Max_Flights_Month  Total_Distance  Avg_Distance_Month  Total_Points_Acc  Total_Points_Red  Total_Dollar_Red  Redemption_Rate  Months_Since_Last_Flight  Zero_Activity_Months  Flights_Fall  Flights_Spring  Flights_Summer  Flights_Winter  Onehot__Gender_Female  Onehot__Gender_Male  Onehot__Marital Status_Divorced  Onehot__Marital Status_Married  Onehot__Marital Status_Single  Onehot__Enrollment Type_2018 Promotion  Onehot__Enrollment Type_Standard  Onehot__Region_East  Onehot__Region_North  Onehot__Region_West  loyalty_card_encoded  Education_encoded  Flight_consistency_score
0   83236.0  3839.14             34        0             37           1.541667                  5           54525         2271.875000           54525.0              1418               256         0.026006                       0.0                  10.0             8              10              16               3                    1.0   

In [157]:
df.to_csv('modelling2.csv')